# 01 — Cleaning (Part 1: Data Inspection)

**Phase A — understand the real dataset before changing anything.**

The Kaggle file is not committed (see `.gitignore`). If
`data/GlobalWeatherRepository.csv` is missing, the notebook downloads it
automatically through KaggleHub (`src/load_data.py`). A manual download also
works; see `data/README.md`.

This part only **inspects** the raw data:

1. load the CSV unchanged,
2. report shape, dtypes, missingness and duplicates,
3. find and parse the real time column,
4. check location columns, numeric ranges, redundant unit pairs,
5. flag physically impossible values.

No cleaning decisions are made here. Phase B adds the cleaning cells to this
same notebook, using the facts produced below.


## 0. Imports and paths

We add the project root to `sys.path` so the notebook can import the reusable
helpers in `src/`. Paths are defined once in `src/config.py`; nothing is
hard-coded in the notebook.


In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

# The notebook normally runs from notebooks/; add the project root so
# `from src...` imports work from either notebooks/ or the project root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_CSV
from src.load_data import ensure_raw_csv
from src.inspect_data import (
    column_overview,
    constant_columns,
    coverage_details,
    detect_location_columns,
    duplicate_summary,
    location_summary,
    missing_table,
    numeric_summary,
    pick_time_column,
    plausibility_flags,
    time_coverage,
    unit_duplicate_pairs,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 60)

print("Project root:", PROJECT_ROOT)
print("Raw CSV expected at:", RAW_CSV)


Project root: D:\PMA\pma-data-analytics
Raw CSV expected at: D:\PMA\pma-data-analytics\data\GlobalWeatherRepository.csv


## 1. Load the raw file

`ensure_raw_csv()` returns the local CSV, downloading it once through
KaggleHub if needed. `pd.read_csv` is then called **without** `parse_dates`
or dtype overrides. Guessing
types at load time can silently turn bad values into `NaN` or change units, so
we inspect the stored values first.


In [2]:
# Reuse data/GlobalWeatherRepository.csv or download it once via KaggleHub.
# The copy is the unmodified Kaggle file; data/ is git-ignored.
RAW_CSV = ensure_raw_csv()

# Read exactly as stored: no date parsing, no type coercion yet.
df = pd.read_csv(RAW_CSV, low_memory=False)

print("Raw CSV:", RAW_CSV)
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns")
print(f"Memory (deep): {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")


Raw CSV: D:\PMA\pma-data-analytics\data\GlobalWeatherRepository.csv
Loaded 170,928 rows x 41 columns


Memory (deep): 158.5 MB


## 2. Structure and column types

`df.info()` shows the stored dtype and non-null count per column. The overview
table adds cardinality and one example value per column, which quickly reveals
IDs, constants, and columns that should be numeric but were read as text.


In [3]:
df.info()
display(column_overview(df))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 170928 entries, 0 to 170927
Data columns (total 41 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   country                       170928 non-null  object 
 1   location_name                 170928 non-null  object 
 2   latitude                      170928 non-null  float64
 3   longitude                     170928 non-null  float64
 4   timezone                      170928 non-null  object 
 5   last_updated_epoch            170928 non-null  int64  
 6   last_updated                  170928 non-null  object 
 7   temperature_celsius           170928 non-null  float64
 8   temperature_fahrenheit        170928 non-null  float64
 9   condition_text                170928 non-null  object 
 10  wind_mph                      170928 non-null  float64
 11  wind_kph                      170928 non-null  float64
 12  wind_degree                   170928 non-nul

,column,dtype,non_null,nulls,null_%,n_unique,example
0,country,object,170928,0,0.0,211,Afghanistan
1,location_name,object,170928,0,0.0,268,Kabul
2,latitude,float64,170928,0,0.0,422,34.52
3,longitude,float64,170928,0,0.0,428,69.18
4,timezone,object,170928,0,0.0,201,Asia/Kabul
5,last_updated_epoch,int64,170928,0,0.0,1575,1715849100
6,last_updated,object,170928,0,0.0,28420,2024-05-16 13:15
7,temperature_celsius,float64,170928,0,0.0,709,26.6
8,temperature_fahrenheit,float64,170928,0,0.0,1180,79.8
9,condition_text,object,170928,0,0.0,58,Partly Cloudy


## 3. First rows

Transposing five rows makes all columns readable without horizontal scrolling.
This is a sanity check, not an assumption about the schema.


In [4]:
df.head(5).T


,0,1,2,3,4
country,Afghanistan,Albania,Algeria,Andorra,Angola
location_name,Kabul,Tirana,Algiers,Andorra La Vella,Luanda
latitude,34.52,41.33,36.76,42.5,-8.84
longitude,69.18,19.82,3.05,1.52,13.23
timezone,Asia/Kabul,Europe/Tirane,Africa/Algiers,Europe/Andorra,Africa/Luanda
last_updated_epoch,1715849100,1715849100,1715849100,1715849100,1715849100
last_updated,2024-05-16 13:15,2024-05-16 10:45,2024-05-16 09:45,2024-05-16 10:45,2024-05-16 09:45
temperature_celsius,26.6,19.0,23.0,6.3,26.0
temperature_fahrenheit,79.8,66.2,73.4,43.3,78.8
condition_text,Partly Cloudy,Partly cloudy,Sunny,Light drizzle,Partly cloudy


## 4. Missing values

Missingness is reported per column. We do not fill anything yet; the amount
and pattern decide whether imputation is defensible in Phase B.


In [5]:
miss = missing_table(df)
display(miss if not miss.empty else "No missing values found.")
print("Rows with at least one missing value:", int(df.isna().any(axis=1).sum()))


'No missing values found.'

Rows with at least one missing value: 0


## 5. Duplicates

Two checks matter:

- **Exact duplicate rows** — the same record stored twice.
- **Entity + timestamp duplicates** — the same city with two readings at the
  same timestamp.

Many rows sharing a timestamp is normal when each city is sampled once. It is
the **country + city + timestamp** key that should be unique, because city
names such as "San Jose" repeat across countries.


In [6]:
time_col = pick_time_column(df)
location_cols = detect_location_columns(df)

# City names repeat across countries ("San Jose", "San Juan", ...), so the
# unique place key is country + city, not city alone.
entity_cols = []
if location_cols["country"]:
    entity_cols.append(location_cols["country"][0])
if location_cols["city"]:
    entity_cols.append(location_cols["city"][0])

print("Time column selected:", time_col)
print("Entity columns selected:", entity_cols)

dups = duplicate_summary(df, time_col=time_col, entity_cols=entity_cols)
dups


Time column selected: last_updated
Entity columns selected: ['country', 'location_name']


{'exact_duplicate_rows': 0,
 'duplicate_timestamp_rows': 142508,
 'duplicate_entity_timestamp_rows': 1}

In [7]:
if dups["exact_duplicate_rows"]:
    display(df[df.duplicated(keep=False)].head(5))
else:
    print("No exact duplicate rows.")


No exact duplicate rows.


## 6. Time column and coverage

`pick_time_column` only trusts a column when at least 95% of its values parse.
We then check the real date range, the number of unique timestamps, missing
calendar days, and the typical gap between readings for a few cities.

This is what tells us how much history exists and whether the series is daily
or sub-daily. Both facts constrain the forecasting design.


In [8]:
display(time_coverage(df))


,column,dtype,parse_%,n_unique,min,max,span_days
0,last_updated,object,100.0,28420,2024-05-16 01:45:00,2026-10-13 18:45:00,880


In [9]:
if time_col is None:
    print("No usable time column found (>=95% parseable).")
else:
    print(f"Time column: {time_col}\n")
    print("\n".join(coverage_details(df, time_col)))


Time column: last_updated



Parsed values: 170,928 / 170,928
Range: 2024-05-16 01:45:00  ->  2026-10-13 18:45:00
Span: 880 days
Unique timestamps: 28,420
Rows per calendar day: min=1, median=195, max=378, days=880
Calendar days with zero rows: 1
Most common timestamps: 2026-01-02 08:15:00 (45 rows), 2026-01-03 08:00:00 (45 rows), 2025-12-26 08:15:00 (45 rows), 2026-01-05 08:15:00 (45 rows), 2026-04-22 08:30:00 (45 rows)
Cadence for 'Kabul' (878 rows): 1 d x 395, 0.99 d x 214, 1.01 d x 172
Cadence for 'Tirana' (877 rows): 1 d x 381, 0.99 d x 212, 1.01 d x 173
Cadence for 'Algiers' (876 rows): 1 d x 395, 0.99 d x 201, 1.01 d x 163

## 7. Location columns

The number of unique cities and countries determines whether a single global
average is meaningful or whether per-city models are needed. Latitude and
longitude are also checked here for later maps.


In [10]:
display(location_summary(df))


,role,column,non_null,n_unique,min,max,top_values
0,city,location_name,170928,268,,,"Sanaa, Tashkent, Bujumbura, Kyiv, N'djamena"
1,country,country,170928,211,,,"Bulgaria, Indonesia, Thailand, Turkey, Sudan"
2,latitude,latitude,170928,422,-41.3,65.3,
3,longitude,longitude,170928,428,-175.2,179.22,


## 8. Numeric distributions

The 1% and 99% quantiles expose extreme values without being as fragile as
min/max. Negative counts and zero counts are included because they often reveal
sentinels (for example `-1` for a missing reading).


In [11]:
display(numeric_summary(df))


,count,mean,std,min,1%,25%,50%,75%,99%,max,negative,zeros
latitude,170928.0,1.928300e+01,2.437300e+01,-4.130000e+01,-3.528000e+01,4.050000e+00,1.725000e+01,4.040000e+01,6.017600e+01,6.530000e+01,37139,0
longitude,170928.0,2.180500e+01,6.576700e+01,-1.752000e+02,-1.717300e+02,-6.836000e+00,2.143300e+01,4.988200e+01,1.784170e+02,1.792200e+02,47801,0
last_updated_epoch,170928.0,1.753891e+09,2.196145e+07,1.715849e+09,1.716473e+09,1.734949e+09,1.753863e+09,1.772865e+09,1.791179e+09,1.791870e+09,0,0
temperature_celsius,170928.0,2.144500e+01,9.272000e+00,-2.980000e+01,-3.900000e+00,1.620000e+01,2.360000e+01,2.770000e+01,3.920000e+01,7.930000e+01,3718,111
temperature_fahrenheit,170928.0,7.060300e+01,1.668900e+01,-2.160000e+01,2.500000e+01,6.110000e+01,7.440000e+01,8.180000e+01,1.025000e+02,1.747000e+02,187,9
wind_mph,170928.0,7.872000e+00,6.827000e+00,2.200000e+00,2.200000e+00,3.800000e+00,6.500000e+00,1.070000e+01,2.300000e+01,1.841200e+03,0,0
wind_kph,170928.0,1.267300e+01,1.098300e+01,3.600000e+00,3.600000e+00,6.100000e+00,1.040000e+01,1.730000e+01,3.710000e+01,2.963200e+03,0,0
wind_degree,170928.0,1.701470e+02,1.032630e+02,1.000000e+00,6.000000e+00,8.100000e+01,1.640000e+02,2.560000e+02,3.560000e+02,3.600000e+02,0,0
pressure_mb,170928.0,1.014037e+03,9.872000e+00,7.710000e+02,9.960000e+02,1.010000e+03,1.014000e+03,1.018000e+03,1.031000e+03,3.006000e+03,0,0
pressure_in,170928.0,2.994400e+01,2.910000e-01,2.276000e+01,2.941000e+01,2.983000e+01,2.993000e+01,3.006000e+01,3.046000e+01,8.877000e+01,0,0


## 9. Constant and redundant columns

Two checks:

- **Constant columns** — zero or one distinct value, so no information.
- **Near-perfect numeric pairs** — columns with |r| >= 0.9999 and a linear
  relationship. A slope near 1.8 with intercept 32 is Celsius vs Fahrenheit; a
  slope near 1.609 is km vs miles. These pairs are candidates for removal to
  avoid duplicate information.


In [12]:
const = constant_columns(df)
print("Constant columns:")
display(const if const else "None.")

pairs = unit_duplicate_pairs(df)
display(pairs if not pairs.empty else "No near-perfect linear pairs found (|r| >= 0.9999).")


Constant columns:


'None.'

,column_a,column_b,n,r,slope,intercept,note
0,temperature_celsius,temperature_fahrenheit,170928,0.999997,1.799981,32.0019,F = 1.8 * C + 32 (Celsius vs Fahrenheit)
1,feels_like_celsius,feels_like_fahrenheit,170928,0.999996,1.799814,31.9997,F = 1.8 * C + 32 (Celsius vs Fahrenheit)
2,gust_mph,gust_kph,170928,0.999992,1.609317,0.0022,km = 1.609344 * miles
3,wind_mph,wind_kph,170928,0.999987,1.608787,0.0082,km = 1.609344 * miles


## 10. Impossible-value flags

These are **heuristics**, not final cleaning rules. Each flag names the exact
rule and the number of affected rows, so every rule can be reviewed and
justified in Phase B.


In [13]:
flags = plausibility_flags(df)
display(flags if not flags.empty else "No heuristic violations found. Manual review is still required.")


,column,rule,bad_rows,examples
0,temperature_celsius,temperature outside -90..60 C,1,79.3
1,temperature_fahrenheit,temperature outside -130..140 F,1,174.7
2,pressure_mb,pressure outside 800-1100 hPa,5,"3006.0, 3000.0, 771.0"
3,pressure_in,pressure outside 20-35 inHg,2,"88.77, 88.59"
4,air_quality_Carbon_Monoxide,negative air-quality value,1,-9999.0
5,air_quality_Sulphur_dioxide,negative air-quality value,1,-9999.0
6,air_quality_PM10,negative air-quality value,2,"-1848.15, -998.15"


## 11. Initial observations (from the executed output above)

- **Rows and columns:** 170,928 ? 41.
- **Stored dtypes needing correction:** `last_updated` and the sun/moon time
  fields are strings; `last_updated_epoch` is an integer UTC epoch. Numeric
  weather fields already load as numbers.
- **Date range and span:** `2024-05-16 01:45:00` ? `2026-10-13 18:45:00`;
  880-day span, 28,420 unique timestamps.
- **Locations:** 268 unique city names, 211 country labels, and 286
  country+city groups. City names repeat across countries, so the entity key
  is **country + city**.
- **Cadence:** roughly daily per city, but not synchronised and highly
  uneven. Rows per day: min 1, median 195, max 378. 183 groups have 800?879
  rows; 30 groups have only one row. There are also 417 country+city+local-date
  keys with 2 readings (max 2), so a daily series needs an explicit aggregation
  rule before modelling.
- **Missing values:** pandas reports zero nulls, but there are sentinel /
  impossible values that must be treated as missing in Phase B.
- **Exact duplicates:** 0 rows. **Country+city+timestamp duplicates:** 1
  (Thailand / Nan, `2026-02-20 14:00`); the duplicate pair differs in
  sunrise/sunset/moonrise/moonset/moon illumination.
- **Constant columns:** none.
- **Unit-duplicate pairs:** temperature C/F, feels-like C/F, wind mph/kph,
  gust mph/kph. The pressure mb/in, precipitation mm/in, and visibility
  km/mile pairs are also unit duplicates by name and conversion (their
  correlations are slightly below the strict near-perfect threshold because
  of rounding).
- **Impossible / extreme values:** Suva `temperature_celsius = 79.3` (and
  feels-like 81.3); pressure 3006 mb (Tegucigalpa) and 3000 mb (Tehran);
  four wind rows above 200 kph, including 2963.2 kph (Bujumbura); air-quality
  sentinels `-9999` (CO, SO2) and negative PM10 values. Pressure 771 mb
  (Addis Ababa) and 787 mb (Mexico City) are plausibly altitude-related, not
  automatically errors.
- **Special time strings:** `moonrise` / `moonset` contain values such as
  `No moonrise`, `Does not rise today`, `....`, `****`, and `24:00`, which
  must be treated as missing/flagged rather than parsed as clock times.
- **Phase B implications:** deduplicate the one repeated key, drop redundant
  unit columns, convert sentinels to missing, flag rather than blindly delete
  extreme weather, reconcile corrupted country/city aliases, and make an
  explicit local-vs-UTC decision for daily aggregation. Per-city forecasting
  is not defensible for the ~30 one-row groups.

## Phase A checkpoint

Phase A is complete. No cleaning, outlier treatment, scaling, EDA, or modelling
has been performed in this notebook yet.


## Phase A checkpoint

Before moving on:

1. Run every cell and save the notebook **with outputs**.
2. Check that `reports/data_profile.txt` was created (the script version of
   this profile is useful for review; it also records the file's SHA-256).
3. Share the key output with the reviewer (shape, date range, cities,
   missingness, duplicates, unit pairs, flags).

No cleaning is performed in this part. Phase B will use these observed facts.


# Part 2 ? Cleaning, Outliers, and Scaling (Phase B)

Part 1 profiled the raw file. This part applies only rules justified by that
profile and records the actual number of rows/cells affected.

The raw CSV is never overwritten. Cleaned outputs go to `data/processed/`,
which is git-ignored.


## 12. Cleaning rules applied

1. Parse `last_updated` as the local timestamp; derive `timestamp_utc` from
   `last_updated_epoch`; create local and UTC dates.
2. Use **country + city** as the entity key, with documented country/city
   aliases for translations, typos, and obvious duplicate spellings.
3. Remove the one duplicate country+city+timestamp row.
4. Set only clearly impossible values to missing and keep an audit flag:
   temperature outside ?90?60 ?C, pressure outside 650?1100 mb, wind/gust
   above 500 kph, and negative air-quality pollutant values.
5. Keep real weather extremes as diagnostic outliers; do not delete them.
6. Drop redundant unit columns and normalise condition casing.
7. Build a city-day table (local date) and an equal-weighted global daily core
   table (UTC date). The global table flags dates with fewer than 150 of 183
   core cities.


In [14]:
from src.clean_data import (
    MODEL_FEATURE_COLUMNS,
    build_daily_city,
    build_daily_global_core,
    clean_rows,
    outlier_by_city,
    outlier_summary,
    save_outputs,
    scaling_summary,
)

cleaned, clean_stats = clean_rows(df)

print(f"Rows: {clean_stats['rows_input']:,} -> {clean_stats['rows_output']:,}")
print(f"Columns: {clean_stats['columns_input']} -> {clean_stats['columns_output']}")
print(f"Duplicate key rows removed: {clean_stats['duplicate_key_rows']}")
print(f"Null cells: {clean_stats['null_cells_input']:,} -> {clean_stats['null_cells_output']:,}")
print("Country labels:", clean_stats["country_unique_raw"], "->", clean_stats["country_unique_clean"])
print("City names:", clean_stats["city_unique_raw"], "->", clean_stats["city_unique_clean"])
print("Country+city groups:", clean_stats["groups_raw"], "->", clean_stats["groups_clean"])


Rows: 170,928 -> 170,927
Columns: 41 -> 48
Duplicate key rows removed: 1
Null cells: 0 -> 10
Country labels: 211 -> 191
City names: 268 -> 262
Country+city groups: 286 -> 269


In [15]:
tier_table = pd.DataFrame({
    "rows": pd.Series(clean_stats["tier_rows"]),
    "groups": pd.Series(clean_stats["tier_groups"]),
}).loc[["core", "partial", "sparse"]]
display(tier_table)

flag_table = pd.Series({
    "temperature impossible": clean_stats["temperature_impossible_rows"],
    "pressure impossible": clean_stats["pressure_impossible_rows"],
    "wind impossible (>500 kph)": clean_stats["wind_impossible_rows"],
    "extreme wind (>200 kph, kept)": clean_stats["extreme_wind_rows"],
    "negative air-quality rows": clean_stats["air_quality_negative_rows"],
    "zero visibility (kept)": clean_stats["visibility_zero_rows"],
}, name="rows")
display(flag_table.to_frame())


,rows,groups
core,160285,183
partial,10007,33
sparse,635,53


,rows
temperature impossible,1
pressure impossible,2
wind impossible (>500 kph),1
"extreme wind (>200 kph, kept)",4
negative air-quality rows,4
zero visibility (kept),1357


In [16]:
serious_flags = [
    "flag_temperature_impossible",
    "flag_pressure_impossible",
    "flag_wind_impossible",
    "flag_air_quality_negative",
]
serious = cleaned.loc[cleaned[serious_flags].any(axis=1),
    ["country", "city", "last_updated", "temperature_celsius", "pressure_mb",
     "wind_kph", "gust_kph", "air_quality_Carbon_Monoxide",
     "air_quality_Sulphur_dioxide", "air_quality_PM10", "quality_flag_count"],
]
print("Rows with an impossible-value flag:", len(serious))
display(serious)
print("Rows with zero visibility (kept; excluded from the table above):",
      int(cleaned.flag_visibility_zero.sum()))


Rows with an impossible-value flag: 8


,country,city,last_updated,temperature_celsius,pressure_mb,wind_kph,gust_kph,air_quality_Carbon_Monoxide,air_quality_Sulphur_dioxide,air_quality_PM10,quality_flag_count
5263,Antigua and Barbuda,Saint John's,2024-06-11 10:00:00,30.3,1017.0,16.9,38.4,NaN,0.400,12.100,1
7601,Burundi,Bujumbura,2024-06-23 15:45:00,22.5,1014.0,NaN,NaN,277.00,0.200,7.200,2
12078,Kiribati,Tarawa,2024-07-17 00:45:00,28.6,1008.0,19.1,25.7,223.60,NaN,0.900,1
49769,Honduras,Tegucigalpa,2025-01-28 05:00:00,19.1,NaN,10.4,18.3,329.30,1.480,8.140,1
52114,Iran,Tehran,2025-02-09 14:00:00,2.1,NaN,8.3,9.5,3146.85,56.610,52.725,1
55303,Saudi Arabia,Riyadh,2025-02-25 13:00:00,10.1,1027.0,21.6,24.8,432.90,28.675,NaN,1
120193,Saudi Arabia,Riyadh,2026-01-25 10:00:00,15.2,1019.0,40.7,46.8,181.85,23.050,NaN,1
137275,Fiji,Suva,2026-04-24 18:30:00,NaN,1013.0,16.2,27.1,75.85,1.350,5.950,1


Rows with zero visibility (kept; excluded from the table above): 1357


In [17]:
outliers = outlier_summary(cleaned)
by_city_outliers = outlier_by_city(cleaned)
scaling = scaling_summary(cleaned)

print("Global IQR outlier counts (diagnostic only):")
display(outliers[["column", "iqr_outliers", "z3_outliers", "min", "max"]])
print("Largest within-city temperature outlier shares:")
display(by_city_outliers.head(8))


Global IQR outlier counts (diagnostic only):


,column,iqr_outliers,z3_outliers,min,max
0,temperature_celsius,3124,1156,-29.80,49.200
1,feels_like_celsius,2890,1119,-40.00,51.200
2,wind_kph,3221,1573,3.60,272.200
3,wind_degree,0,0,1.00,360.000
4,gust_kph,4493,1835,3.60,279.400
5,pressure_mb,4572,1247,771.00,1080.000
6,precip_mm,33887,2050,0.00,74.520
7,humidity,2,0,2.00,100.000
8,cloud,0,0,0.00,100.000
9,visibility_km,34808,5144,0.00,32.000


Largest within-city temperature outlier shares:


,country,city,n,temperature_outliers,temperature_share,wind_outliers,wind_share,pressure_outliers,pressure_share
153,Suriname,Paramaribo,875,115,0.131429,7,0.008000,1,0.001143
67,Haiti,Port-Au-Prince,875,99,0.113143,61,0.069714,28,0.032000
133,Saint Lucia,Castries,877,95,0.108324,2,0.002281,0,0.000000
20,Bolivia,National,872,90,0.103211,80,0.091743,14,0.016055
66,Guyana,Georgetown,874,82,0.093822,10,0.011442,5,0.005721
163,Trinidad and Tobago,Port Of Spain,875,80,0.091429,69,0.078857,5,0.005714
99,Marshall Islands,Majuro,878,70,0.079727,0,0.000000,2,0.002278
24,Brazil,Bras,872,63,0.072248,70,0.080275,1,0.001147


In [18]:
daily_city = build_daily_city(cleaned)
daily_global = build_daily_global_core(cleaned)

print(f"Daily city rows: {len(daily_city):,}")
print(f"City-day keys with 2 source readings: {int((daily_city.n_readings == 2).sum()):,}")
print(f"Global core UTC dates: {len(daily_global):,}")
print(f"Usable global dates (>=150 cities): {int(daily_global.coverage_ok.sum()):,}")
display(daily_global.head(3))


Daily city rows: 170,511
City-day keys with 2 source readings: 416
Global core UTC dates: 879
Usable global dates (>=150 cities): 875


,date_utc,temperature_celsius,feels_like_celsius,wind_kph,gust_kph,pressure_mb,precip_mm,humidity,cloud,visibility_km,uv_index,moon_illumination,air_quality_Carbon_Monoxide,air_quality_Ozone,air_quality_Nitrogen_dioxide,air_quality_Sulphur_dioxide,air_quality_PM2.5,air_quality_PM10,n_cities,coverage_ok
0,2024-05-16,23.843370,25.470994,14.542541,21.969061,1013.378453,0.135552,62.212707,38.602210,9.847790,4.919890,55.0,540.145307,59.913812,11.610773,8.493646,19.109669,34.129006,181,True
1,2024-05-17,24.528492,26.356983,13.586034,22.190503,1012.927374,0.106592,60.469274,32.927374,9.720112,3.893855,64.0,566.288821,57.696648,14.743017,7.961453,19.636872,35.745251,179,True
2,2024-05-18,25.958101,28.312849,13.617318,21.550279,1012.748603,0.136648,56.564246,31.530726,9.826816,5.262570,73.0,471.771508,64.783799,8.439106,5.284916,17.018436,34.378771,179,True


In [19]:
from sklearn.preprocessing import StandardScaler

# Demonstration of leakage-safe scaling ONLY. The final train/test split and
# modelling horizon are chosen in Phase C.
demo = daily_global.loc[daily_global["coverage_ok"]].copy()
demo["date_utc"] = pd.to_datetime(demo["date_utc"])
demo = demo.sort_values("date_utc").reset_index(drop=True)
cut_index = int(len(demo) * 0.80)
cut_date = demo.loc[cut_index, "date_utc"]
train = demo.loc[demo["date_utc"] < cut_date]
test = demo.loc[demo["date_utc"] >= cut_date]
features = [c for c in MODEL_FEATURE_COLUMNS if c in demo.columns]

scaler = StandardScaler().fit(train[features])
train_z = scaler.transform(train[features])
test_z = scaler.transform(test[features])

scale_check = pd.DataFrame({
    "feature": features,
    "raw_train_mean": train[features].mean().values,
    "scaler_fit_mean": scaler.mean_,
    "raw_train_std": train[features].std(ddof=0).values,
    "scaler_fit_scale": scaler.scale_,
})
display(scale_check)
print(f"Train dates: {train.date_utc.min().date()} -> {train.date_utc.max().date()} ({len(train)} rows)")
print(f"Test dates:  {test.date_utc.min().date()} -> {test.date_utc.max().date()} ({len(test)} rows)")
print("Max absolute train-scaled mean:", abs(train_z.mean(axis=0)).max().round(10))
print("Max absolute test-scaled mean: ", abs(test_z.mean(axis=0)).max().round(4))
print("Scaler was fitted on training rows only; the test mean is not forced to zero.")


,feature,raw_train_mean,scaler_fit_mean,raw_train_std,scaler_fit_scale
0,feels_like_celsius,22.126646,22.126646,4.568120,4.568120
1,wind_kph,12.964620,12.964620,1.052634,1.052634
2,gust_kph,18.289816,18.289816,1.851290,1.851290
3,pressure_mb,1014.109653,1014.109653,1.901382,1.901382
4,precip_mm,0.131607,0.131607,0.046891,0.046891
5,humidity,66.369480,66.369480,4.959920,4.959920
6,cloud,39.521493,39.521493,3.505977,3.505977
7,visibility_km,9.489169,9.489169,0.344090,0.344090
8,uv_index,3.350544,3.350544,1.353913,1.353913
9,air_quality_Carbon_Monoxide,465.353428,465.353428,117.451623,117.451623


Train dates: 2024-05-16 -> 2026-04-21 (700 rows)
Test dates:  2026-04-22 -> 2026-10-13 (175 rows)
Max absolute train-scaled mean: 0.0
Max absolute test-scaled mean:  1.9483
Scaler was fitted on training rows only; the test mean is not forced to zero.


In [20]:
paths = save_outputs(cleaned, daily_city, daily_global, outliers, by_city_outliers, scaling)
for name, path in paths.items():
    print(f"{name}: {path}")


clean_rows: D:\PMA\pma-data-analytics\data\processed\weather_clean_rows.csv.gz
daily_city: D:\PMA\pma-data-analytics\data\processed\weather_daily_city.csv.gz
daily_global: D:\PMA\pma-data-analytics\data\processed\weather_daily_global_core.csv
outliers: D:\PMA\pma-data-analytics\reports\outlier_summary.csv
outlier_city: D:\PMA\pma-data-analytics\reports\outlier_by_city_temperature.csv
scaling: D:\PMA\pma-data-analytics\reports\scaling_summary.csv
aliases: D:\PMA\pma-data-analytics\reports\alias_mapping.csv


## Phase B checkpoint

Phase B is complete for review:

- The raw row count went from 170,928 to 170,927 because one repeated
  country+city+timestamp row was removed.
- Only impossible values were masked (1 temperature row, 2 pressure rows,
  1 wind row, 4 negative air-quality rows); real extremes were retained and
  flagged.
- The city-day table has one row per country+city+local date.
- The global core table has one row per UTC date, with a `coverage_ok` flag.
- Scaling was demonstrated with a scaler fitted on training dates only. No
  model has been trained yet.

No EDA figures, forecast metrics, or model comparisons have been produced.
